In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, current_timestamp, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [4]:
spark = SparkSession.builder \
    .appName("bronze-layer-structured") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(" Spark com Iceberg iniciado!")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c2b204a0-8125-40e3-ba25-f92e66229aba;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
:: resolution report :: resolve 213ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 from central in [default]
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number

 Spark com Iceberg iniciado!


In [5]:
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

DataFrame[]

In [6]:
df_raw = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/pipeline_transacoes") \
    .option("dbtable", "staging.transacoes_raw") \
    .option("user", "postgres") \
    .option("password", "1234") \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [7]:
schema_payload = StructType([
    StructField("valor", DoubleType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("categoria", StringType(), True),
    StructField("metodo_pagamento", StringType(), True)
])

In [8]:
df_bronze = df_raw.withColumn(
    "data_ingestao", current_timestamp()
).withColumn(
    "data_particao", to_date(col("data_ingestao"))
).withColumn(
    "payload_struct", from_json(col("payload"), schema_payload)
).select(
    "id",
    "payload_struct.*", 
    "data_ingestao",
    "data_particao"
)

In [9]:
df_bronze.writeTo("lakehouse.bronze.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()


In [10]:
spark.sql("SELECT * FROM lakehouse.bronze.transacoes LIMIT 5").show()

+---+-------+----------+-----------+----------------+--------------------+-------------+
| id|  valor|cliente_id|  categoria|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+----------+-----------+----------------+--------------------+-------------+
|  1|2321.91|         5|eletronicos|             pix|2026-03-30 00:25:...|   2026-03-30|
|  2|4130.44|        79|  vestuario|          boleto|2026-03-30 00:25:...|   2026-03-30|
|  3|4633.68|        17|eletronicos|          boleto|2026-03-30 00:25:...|   2026-03-30|
|  4| 3685.2|        52|  vestuario|  cartao_credito|2026-03-30 00:25:...|   2026-03-30|
|  5|1817.05|        12|  vestuario|          boleto|2026-03-30 00:25:...|   2026-03-30|
+---+-------+----------+-----------+----------------+--------------------+-------------+



In [11]:
!ls /tmp/warehouse/bronze/transacoes

data  metadata
